In [ ]:
# --- Sección 1: Carga de datos y preparación ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Rutas a los datos (reemplaza con tus rutas reales)
path = r"D:\MulticentersEEG\Features_2_normativeModel\gamma_40\24_BEST_EPOCHS\AIF_Babiloni\results_harmonize\neuroharmonize\osc_pw_ab_canonic_roiC_SITE_age_neuroharmonize.xlsx"
data_harmonized = pd.read_excel(path, sheet_name='unharmonizeSITE_age')
data_unharmonized = pd.read_excel(path, sheet_name='harmonizeSITE_age')

roi_ids = list(data_harmonized.columns[4:-4])

In [ ]:
all_data_features = data_harmonized[roi_ids]
all_data_covariates = data_harmonized[['age','SITE']]

In [24]:
all_data_covariates = pd.get_dummies(all_data_covariates, columns=['SITE'])
all_data_covariates

,age,SITE_Argentina,SITE_Chile,SITE_Cuba,SITE_Dortmund,SITE_Greece,SITE_Madrid,SITE_Medellin_hd,SITE_Medellin_ld,SITE_Oslo,SITE_Poland,SITE_Seoul
0,60,False,False,False,True,False,False,False,False,False,False,False
1,67,False,False,False,True,False,False,False,False,False,False,False
2,44,False,False,False,True,False,False,False,False,False,False,False
3,24,False,False,False,True,False,False,False,False,False,False,False
4,48,False,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
1541,85,False,False,False,False,False,True,False,False,False,False,False
1542,84,False,False,False,False,False,True,False,False,False,False,False
1543,84,False,False,False,False,False,True,False,False,False,False,False
1544,81,False,False,False,False,False,True,False,False,False,False,False


In [25]:
X_train, X_test, y_train, y_test = train_test_split(all_data_covariates, all_data_features, stratify=data_harmonized['SITE'], test_size=0.2, random_state=42)


In [26]:
tr_cov_size = X_train.shape
tr_resp_size = y_train.shape
te_cov_size = X_test.shape
te_resp_size = y_test.shape
print("Train covariate size is: ", tr_cov_size)
print("Test covariate size is: ", te_cov_size)
print("Train response size is: ", tr_resp_size)
print("Test response size is: ", te_resp_size)

Train covariate size is:  (1236, 12)
Test covariate size is:  (310, 12)
Train response size is:  (1236, 9)
Test response size is:  (310, 9)


In [27]:
X_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

In [44]:
DIRECTORY = r"D:\MulticentersEEG\Features_2_normativeModel\gamma_40\24_BEST_EPOCHS\AIF_Babiloni\results_harmonize\neuroharmonize\blr_pcntoolkit/"
for c in y_train.columns:
    y_train[c].to_csv(DIRECTORY+'resp_tr_' + c + '.txt', header=False, index=False)

In [45]:
X_train.to_csv(DIRECTORY+'cov_tr.txt', sep = '\t', header=False, index = False)


In [46]:
y_train.to_csv(DIRECTORY+'resp_tr.txt', sep = '\t', header=False, index = False)


In [47]:
for c in y_test.columns:
    y_test[c].to_csv(DIRECTORY+'resp_te_' + c + '.txt', header=False, index=False)

In [48]:
X_test.to_csv(DIRECTORY+'cov_te.txt', sep = '\t', header=False, index = False)


In [49]:
y_test.to_csv(DIRECTORY+'resp_te.txt', sep = '\t', header=False, index = False)


In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from pcntoolkit.normative import estimate, evaluate
from pcntoolkit.util.utils import create_bspline_basis, compute_MSLL
import os

In [51]:
data_dir = DIRECTORY

# Create a cubic B-spline basis (used for regression)
xmin = 10#16 # xmin & xmax are the boundaries for ages of participants in the dataset
xmax = 95#90
B = create_bspline_basis(xmin, xmax)
# create the basis expansion for the covariates for each of the 
for roi in roi_ids: 
    print('Creating basis expansion for ROI:', roi)
    roi_dir = os.path.join(data_dir, roi)
    os.chdir(roi_dir)
    # create output dir 
    os.makedirs(os.path.join(roi_dir,'blr'), exist_ok=True)
    # load train & test covariate data matrices
    X_tr = np.loadtxt(os.path.join(roi_dir, 'cov_tr.txt'))
    X_te = np.loadtxt(os.path.join(roi_dir, 'cov_te.txt'))
    # add intercept column 
    X_tr = np.concatenate((X_tr, np.ones((X_tr.shape[0],1))), axis=1)
    X_te = np.concatenate((X_te, np.ones((X_te.shape[0],1))), axis=1)
    np.savetxt(os.path.join(roi_dir, 'cov_int_tr.txt'), X_tr)
    np.savetxt(os.path.join(roi_dir, 'cov_int_te.txt'), X_te)
    
    # create Bspline basis set 
    Phi = np.array([B(i) for i in X_tr[:,0]])
    Phis = np.array([B(i) for i in X_te[:,0]])
    X_tr = np.concatenate((X_tr, Phi), axis=1)
    X_te = np.concatenate((X_te, Phis), axis=1)
    np.savetxt(os.path.join(roi_dir, 'cov_bspline_tr.txt'), X_tr)
    np.savetxt(os.path.join(roi_dir, 'cov_bspline_te.txt'), X_te)

Creating basis expansion for ROI: osc_pw_ab_canonic_theta_roiC


FileNotFoundError: D:\MulticentersEEG\Features_2_normativeModel\gamma_40\24_BEST_EPOCHS\AIF_Babiloni\results_harmonize\neuroharmonize\blr_pcntoolkit/osc_pw_ab_canonic_theta_roiC\cov_tr.txt not found.

In [ ]:
# Create pandas dataframes with header names to save out the overall and per-site model evaluation metrics
blr_metrics = pd.DataFrame(columns = ['ROI', 'MSLL', 'EV', 'SMSE', 'RMSE', 'Rho'])
blr_site_metrics = pd.DataFrame(columns = ['ROI', 'site', 'MSLL', 'EV', 'SMSE', 'RMSE', 'Rho'])